# 01 · Татарский Whisper → транскрипция → ручная вычитка → TTS-корпус

**Что будет сделано:** 4 MP3 → черновые временные метки Whisper → CSV/SRT → ручная проверка → сегменты WAV 22.05 кГц + `metadata.csv`.

**ВАЖНО:** говорящий **не** определяется автоматически; у всех черновых фрагментов `approved=0`, `speaker_id` пуст. Нельзя обучать голос по ним до проверки. Музыку и других собеседников исключать вручную.

Вход: разместите исходные MP3 в папке `Мой диск/tatar_audio_lab/raw_mp3/`. Все промежуточные результаты и файлы проверки сохраняются в Drive. Повторный запуск продолжает с ранее завершённых 3-минутных частей. Модель ASR: `yasalma/whisper-finetuned-tt-asr`. Экспериментальная альтернатива для чистого чтения: `yasalma/whisper-finetuned-tatartts-asr`.

**Точность:** Whisper даёт черновик, а не эталон; временные границы слов могут ошибаться. Для TTS нужно привести к полному совпадению текста с аудио.

**Права:** создание персонального синтетического голоса допустимо только с необходимыми разрешениями на голос, исполнение и записи. Для актёрских записей нельзя считать это автоматически разрешённым из-за некоммерческого статуса.
**Версия FIXED:** обход несовместимого токенизатора (с использованием `openai/whisper-small`), режим `SMOKE_TEST=True`. После успешного теста установите `SMOKE_TEST=False`. Проверка синтаксиса выполнена; исполнение в Colab ещё не проверено.


In [ ]:
import os, sys, time, json, csv, re, shutil, subprocess, math
from pathlib import Path
import torch
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (можно, но долго)')
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# Нужен интернет Colab; установка не выполняется в локальной среде ChatGPT.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.45,<5', 'accelerate>=0.33', 'soundfile>=0.12',
                'pandas>=2.0', 'librosa>=0.10', 'tqdm', 'huggingface_hub'], check=True)
subprocess.run(['ffmpeg','-version'], check=True, stdout=subprocess.DEVNULL)
print('Установлены зависимости ASR.')


In [ ]:
from google.colab import drive, files
from IPython.display import display, Audio
import pandas as pd
from tqdm.auto import tqdm
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/tatar_audio_lab')
RAW=ROOT/'raw_mp3'
ASR=ROOT/'asr'
WORK=Path('/content/tatar_asr_work')
for p in [RAW,ASR,WORK]: p.mkdir(parents=True,exist_ok=True)
print('Папка для четырёх исходников:', RAW)
print('Найдено MP3:', [p.name for p in RAW.glob('*.mp3')])


**Если MP3 ещё не загружены в Drive:** можно выполнить следующую ячейку, предварительно выставив `UPLOAD_NOW=True`. Лучше загрузить почти 202 МБ через интерфейс Google Drive, чтобы повторные сессии не требовали новой передачи.

In [ ]:
UPLOAD_NOW = False
if UPLOAD_NOW:
    for name, blob in files.upload().items():
        if name.lower().endswith('.mp3'):
            dest=RAW/name
            dest.write_bytes(blob)
            print('Сохранено:',dest)


In [ ]:
def probe_duration(path):
    data=json.loads(subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration,bit_rate:stream=sample_rate,channels',
        '-of','json',str(path)], text=True))
    s=data['streams'][0]
    return float(data['format']['duration']), int(s['sample_rate']), int(s['channels'])

# Если файл подкаста оказался слишком велик для загрузки через ChatGPT в Drive,
# здесь восстанавливаем его из двух частей, НЕ смешивая их с прочими источниками.
parts = [RAW/'podcast_part01.mp3', RAW/'podcast_part02.mp3']
source_files = [p for p in RAW.glob('*.mp3') if p.name not in {x.name for x in parts}]
if all(p.exists() for p in parts) and not any(p.name.startswith('Подкаст #3') for p in source_files):
    target = WORK/'Подкаст #3 Ринат Тазетдинов.mp3'
    if not target.exists():
        concatlist=WORK/'podcast_concat.txt'
        concatlist.write_text(''.join("file '"+str(p)+"'\n" for p in parts),encoding='utf-8')
        subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-f','concat','-safe','0',
                        '-i',str(concatlist),'-c','copy',str(target)],check=True)
    source_files.append(target)
    print('Подкаст реконструирован в Colab из 2 частей:',target)
sources={f'audio_{i:02d}':p for i,p in enumerate(sorted(source_files),1)}
assert sources, f'Сначала загрузите MP3 в {RAW}'
manifest=[]
for sid,p in sources.items():
    d,sr,ch=probe_duration(p)
    manifest.append({'source_id':sid,'file':p.name,'seconds':round(d,2),'sample_rate':sr,'channels':ch,'status':'unreviewed'})
manifest_df=pd.DataFrame(manifest)
display(manifest_df)
manifest_df.to_csv(ROOT/'source_manifest.csv', index=False, encoding='utf-8-sig')


In [ ]:
# Обход несовместимости tokenizer_config.json татарского чекпойнта с transformers v4.
# Базовая модель checkpoint: openai/whisper-small (см. карточку модели на Hugging Face).
# Токенизатор берём у базовой модели, веса распознавателя — у татарской fine-tuned.
from transformers import pipeline, AutoTokenizer, AutoFeatureExtractor, AutoConfig
MODEL_ID = 'yasalma/whisper-finetuned-tt-asr'
BASE_TOKENIZER = 'openai/whisper-small'
DEVICE = 0 if torch.cuda.is_available() else -1
DTYPE = torch.float16 if DEVICE == 0 else torch.float32
if DEVICE == -1:
    print('⚠ GPU выключен. Для обработки 2 часов звука откройте Runtime -> Change runtime type -> T4 GPU.')
# В transformers 4.x extra_special_tokens ожидается как dict. Ставим корректный
# тип явно, не загружая сломанный tokenizer_config.json из fine-tuned checkpoint.
tokenizer = AutoTokenizer.from_pretrained(BASE_TOKENIZER, extra_special_tokens={})
feature_extractor = AutoFeatureExtractor.from_pretrained(BASE_TOKENIZER)
config = AutoConfig.from_pretrained(MODEL_ID)
if len(tokenizer) != config.vocab_size:
    raise RuntimeError(f'Размер словаря не совпал: tokenizer={len(tokenizer)}, model={config.vocab_size}.')
recognizer = pipeline('automatic-speech-recognition', model=MODEL_ID,
                      tokenizer=tokenizer, feature_extractor=feature_extractor,
                      device=DEVICE, dtype=DTYPE,
                      chunk_length_s=25, stride_length_s=(4,4),
                      batch_size=1 if DEVICE == -1 else 4)
print('Whisper loaded:', MODEL_ID, 'base tokenizer:', BASE_TOKENIZER)


### Автоматическое распознавание с возобновлением

Каждый исходник режется на независимые части по 180 секунд для надёжного сохранения прогресса; Whisper внутри них использует окна со сдвигом. Все результаты черновые. Для первого теста задайте `SMOKE_TEST=True` (только первые 3 минуты каждого файла). Затем `False` — вся коллекция.

In [ ]:
SMOKE_TEST = True  # только первые 3 минуты каждого файла; после успешного теста переключить в False
PART_LENGTH_SEC = 180
GENERATE_WORD_TIMESTAMPS = True

def extract_part(src,start,duration,out):
    subprocess.run(['ffmpeg','-hide_banner','-nostats','-loglevel','error','-y',
                    '-ss',str(start),'-i',str(src),'-t',str(duration),'-vn',
                    '-ac','1','-ar','16000','-c:a','pcm_s16le',str(out)],check=True)

for sid,src in sources.items():
    dur,_,_=probe_duration(src)
    task_dir=ASR/sid
    task_dir.mkdir(parents=True,exist_ok=True)
    count=math.ceil(dur/PART_LENGTH_SEC)
    for j in tqdm(range(count),desc=src.name[:40]):
        if SMOKE_TEST and j>0: break
        target=task_dir/f'part_{j:04d}.json'
        if target.exists(): continue
        offset=j*PART_LENGTH_SEC
        chunk_sec=min(PART_LENGTH_SEC, dur-offset)
        tmp=WORK/'current_asr_part.wav'
        extract_part(src,offset,chunk_sec,tmp)
        try:
            result=recognizer(str(tmp),return_timestamps='word' if GENERATE_WORD_TIMESTAMPS else True,
                              generate_kwargs={'language':'tatar','task':'transcribe'})
        except (ValueError,NotImplementedError) as e:
            if not GENERATE_WORD_TIMESTAMPS: raise
            print('Word timestamps недоступны, fallback на сегменты:',str(e)[:200])
            result=recognizer(str(tmp),return_timestamps=True,
                              generate_kwargs={'language':'tatar','task':'transcribe'})
        data={'source_id':sid, 'source_name':src.name, 'offset_sec':offset,
              'chunk_sec':chunk_sec, 'model':MODEL_ID,'result':result}
        target.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8')
        tmp.unlink(missing_ok=True)
    print(sid,'готово частей:',len(list(task_dir.glob('part_*.json'))),'из',count)


### Выгрузка транскриптов, SRT и черновых пар

Мы группируем слова Whisper в короткие реплики по паузам и пунктуации. Если модель не возвращает временные метки слов, 20–30-секундные сегменты остаются `needs_split` и **не должны попадать в обучающий набор** до разделения.

In [ ]:
def timestamp_srt(sec):
    m=int(sec)//60
    h=m//60
    mi=m%60
    s=sec-int(sec)
    return f'{h:02}:{mi:02}:{s:06.3f}'.replace('.',',')

def pack_utterances(chunks, sid, file_name):
    out=[]; buf=[]
    def flush():
        if not buf: return
        txt=''.join(w['text'] for w in buf).strip()
        start=float(buf[0]['start']); end=float(buf[-1]['end'])
        if txt and end>start:
            out.append({'source_id':sid,'source_file':file_name,'start':round(start,3),
                'end':round(end,3),'text':txt,'speaker_id':'', 'approved':0,
                'quality_notes': 'NEEDS_REVIEW: speech/music/speaker/alignment',
                'status':'needs_split' if end-start>13.0 else 'needs_review'})
        buf.clear()
    for w in chunks:
        if w['end']<=w['start'] or not w['text'].strip(): continue
        if buf and (w['start']-buf[-1]['end']>0.65 or w['end']-buf[0]['start']>11): flush()
        buf.append(w)
        duration=buf[-1]['end']-buf[0]['start']
        if duration>=2.0 and (re.search(r'[.!?…]$',w['text'].strip()) or duration>=10): flush()
    flush()
    return out

rows=[]
for sid,src in sources.items():
    words=[]
    for jp in sorted((ASR/sid).glob('part_*.json')):
        saved=json.loads(jp.read_text(encoding='utf-8'))
        offset=saved['offset_sec']
        for ch in saved['result'].get('chunks',[]):
            start,end=ch.get('timestamp') or (None,None)
            if start is None: continue
            if end is None: end=start+0.3
            words.append({'start':float(start)+offset,'end':float(end)+offset,'text':ch.get('text','')})
    utt=pack_utterances(words,sid,src.name)
    rows.extend(utt)
    srt=[]
    for idx,r in enumerate(utt,1):
        srt.append(f"{idx}\n{timestamp_srt(r['start'])} --> {timestamp_srt(r['end'])}\n{r['text']}\n")
    (ASR/f'{sid}.srt').write_text('\n'.join(srt),encoding='utf-8')
    (ASR/f'{sid}_draft.txt').write_text('\n'.join(r['text'] for r in utt),encoding='utf-8')
    print(sid, 'черновых фраз:',len(utt))
df=pd.DataFrame(rows, columns=['source_id','source_file','start','end','text','speaker_id','approved','quality_notes','status'])
draft=ASR/'reviewed_segments.csv'
if not draft.exists():
    df.to_csv(draft,index=False,encoding='utf-8-sig')
    print('Создан файл для ручной вычитки:',draft)
else:
    print('Существующая ВЫЧИТКА НЕ ПЕРЕЗАПИСАНА:',draft)
display(df.head(15))


In [ ]:
def play_review_row(n, padding=0.3):
    table=pd.read_csv(ASR/'reviewed_segments.csv').fillna('')
    row=table.iloc[int(n)]
    sid=row['source_id']
    start=max(0.0,float(row['start'])-padding)
    duration=float(row['end'])-float(row['start'])+2*padding
    dest=WORK/'listen_review.wav'
    extract_part(sources[sid],start,duration,dest)
    print('index=',n,'source=',sid, 'time=',row['start'],row['end'],'Whisper:',row['text'])
    display(Audio(filename=str(dest)))
# Пример: play_review_row(0)


### После ручной вычитки — нарезка для TTS

Откройте `asr/reviewed_segments.csv` и проверьте каждый фрагмент. Заполните точный `speaker_id` только для разрешённого диктора, поставьте `approved=1` только при уверенности в тексте, временных границах, отсутствии музыки/чужого голоса и пригодном качестве. Исправлять нужно **по реальной записи**, не по оригиналу произведения.

Сначала оформите разрешение на модель конкретного голоса. В следующей ячейке явный флаг предотвращает случайную нарезку актёрских записей для синтеза без разрешения.

In [ ]:
RIGHTS_CONFIRMED_FOR_VOICE_MODEL = False  # True только при наличии необходимых прав/разрешений
AUTHORIZED_SPEAKER_ID = 'consented_speaker_01'
BUILD_DATASET = False  # Включить после проверки и прав

if BUILD_DATASET:
    assert RIGHTS_CONFIRMED_FOR_VOICE_MODEL, 'Не подтверждены права на использование голоса для синтеза.'
    review=pd.read_csv(ASR/'reviewed_segments.csv').fillna('')
    good=review[(review['approved'].astype(str).str.lower().isin(['1','true','yes'])) &
                (review['speaker_id']==AUTHORIZED_SPEAKER_ID)]
    assert len(good)>=10, 'Мало проверенных фрагментов для демонстрации сборки датасета.'
    corpus=ROOT/'corpus_22050'/AUTHORIZED_SPEAKER_ID
    wav_dir=corpus/'wav'; wav_dir.mkdir(parents=True,exist_ok=True)
    metadata=[]; audit=[]
    for i,(_,r) in enumerate(good.iterrows()):
        st=float(r['start']); en=float(r['end']); duration=en-st; text=str(r['text']).strip()
        if not 1.1<=duration<=13.0 or not text or '|' in text: continue
        sid=r['source_id']; src=sources[sid]
        stem=f'{sid}_{i:06d}'
        dest=wav_dir/(stem+'.wav')
        if not dest.exists():
            subprocess.run(['ffmpeg','-hide_banner','-loglevel','error','-y','-ss',str(st),
                            '-i',str(src),'-t',str(duration),'-vn','-ac','1','-ar','22050',
                            '-c:a','pcm_s16le',str(dest)],check=True)
        metadata.append((stem,text))
        audit.append({'id':stem,'source':src.name,'start':st,'end':en,'text':text})
    assert metadata, 'После фильтрации не осталось материала.'
    (corpus/'metadata.csv').write_text(''.join(f'{sid}|{txt}\n' for sid,txt in metadata),encoding='utf-8')
    pd.DataFrame(audit).to_csv(corpus/'audit.csv',index=False,encoding='utf-8-sig')
    print('Сегментов:',len(metadata),'Формат Piper LJSpeech:',corpus)
else:
    print('BUILD_DATASET=False: оставлены только черновики ASR и инструменты ручной проверки.')


**Результат:** `asr/audio_XX.srt` — черновые субтитры; `asr/reviewed_segments.csv` — основная рабочая таблица для ручной проверки; `corpus_22050/consented_speaker_01/wav` + `metadata.csv` — только после ручного отбора и подтверждения прав.

См. руководство в ZIP. В Colab продолжайте следующей сессией: исходные MP3 и накопленные результаты уже находятся в Google Drive.